# Huấn luyện bộ lọc biển số HOG + SVM
Huấn luyện SVM tuyến tính từ đầu trên nhãn khung YOLO có sẵn. Gaussian, Sobel và phép đóng sinh vùng ứng viên. HOG biểu diễn hướng gradient của mỗi vùng. SVM học cách lọc biển thật và vùng nhiễu, không dùng pretrained và chạy trên CPU.

Chạy notebook từ gốc repo với Python có `numpy`, `opencv-python` và `scikit-learn`. Nếu cần có thể cài bằng `%pip install numpy opencv-python scikit-learn`. Đặt `DATASET_OVERRIDE` khi dataset ở đường dẫn khác.

In [ ]:
import csv
import json
import random
from pathlib import Path

import cv2
import numpy as np
from sklearn.svm import LinearSVC

from src.detector import PlateDetector, iou, proposal_features, propose

DATASET_OVERRIDE = None
SMOKE_RUN = False
SEED = 42
MAX_SIDE = 768
IOU_THRESHOLD = 0.5
OUTPUT_DIR = Path('outputs/svm_smoke' if SMOKE_RUN else 'outputs/svm_detector')

dataset_name = 'License Plate Detection Dataset'
roots = [
    Path(dataset_name),
    Path('../CuoiKy') / dataset_name,
]
kaggle_input = Path('/kaggle/input')
if kaggle_input.is_dir():
    roots.extend(sorted(kaggle_input.rglob(dataset_name)))
if DATASET_OVERRIDE is not None:
    roots = [Path(DATASET_OVERRIDE)]
root = next((p for p in roots if (p / 'images').is_dir() and (p / 'labels').is_dir()), None)
if root is None:
    raise FileNotFoundError('Dat DATASET_OVERRIDE den thu muc co images/ va labels/.')
print(root.resolve())
# Cấu hình Sobel đã chọn trong notebook cơ sở.
PROPOSAL = dict(method='sobel', kernel=[9, 3], min_area_ratio=0.002)
MAX_BOXES = 5

def proposals(gray):
    return propose(gray, PROPOSAL, max_side=MAX_SIDE, max_boxes=MAX_BOXES)

## 1. Nhãn và tập dữ liệu
Tập train dùng để học trọng số. Tập val chọn hệ số regularization `C` và ngưỡng SVM. Tập test dùng để so sánh cuối cùng. Danh sách loại trừ là các bản trùng pixel đã xác định trong EDA.

In [ ]:
EXACT_DUPLICATES = {
    'train': {'quandoi64.jpg', 'quandoi92.jpg', 'quandoi99.jpg'},
    'val': set(),
    'test': {'quandoi75.jpg'},
}
IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

def image_paths(directory):
    return sorted(p for p in directory.iterdir() if p.suffix.lower() in IMAGE_SUFFIXES)

def split_paths(split):
    paths = image_paths(root / 'images' / split)
    paths = [p for p in paths if p.name not in EXACT_DUPLICATES[split]]
    if SMOKE_RUN:
        paths = random.Random(SEED).sample(paths, min(18, len(paths)))
    if not paths:
        raise ValueError(f'Tap {split} rong.')
    return paths

def read_gray(path):
    gray = cv2.imdecode(np.fromfile(path, dtype=np.uint8), cv2.IMREAD_GRAYSCALE)
    if gray is None:
        raise ValueError(f'Khong doc duoc: {path}')
    return gray

def ground_truth(path, split, image_shape):
    height, width = image_shape
    relative = path.relative_to(root / 'images' / split).with_suffix('.txt')
    label_path = root / 'labels' / split / relative
    boxes = []
    for row in label_path.read_text(encoding='utf-8-sig').splitlines():
        if not row.strip():
            continue
        label, x, y, w, h = map(float, row.split())
        if label != 0 or w <= 0 or h <= 0:
            raise ValueError(f'Nhan khong hop le: {label_path}')
        box = (
            max(0, (x - w / 2) * width),
            max(0, (y - h / 2) * height),
            min(width, (x + w / 2) * width),
            min(height, (y + h / 2) * height),
        )
        if box[0] >= box[2] or box[1] >= box[3]:
            raise ValueError(f'Khung nam ngoai anh: {label_path}')
        boxes.append(box)
    return boxes

train_paths = split_paths('train')
val_paths = split_paths('val')
test_paths = split_paths('test')
print(f'Train: {len(train_paths)} | Val: {len(val_paths)} | Test: {len(test_paths)}')

## 2. Tạo đặc trưng và huấn luyện
Các crop theo nhãn thật bổ sung ví dụ dương. Ứng viên có IoU ≥ 0,5 với nhãn là dương, IoU ≤ 0,2 là âm. Bỏ qua vùng ở giữa để giảm nhãn mơ hồ. Dùng trọng số cân bằng hai lớp. Giá trị SVM là điểm phân loại có dấu.

In [ ]:
features, classes = [], []
for path in train_paths:
    gray = read_gray(path)
    truth = ground_truth(path, 'train', gray.shape)
    for box in truth:
        features.append(proposal_features(gray, box))
        classes.append(1)
    for box, _ in proposals(gray):
        overlap = max((iou(box, ref) for ref in truth), default=0)
        if overlap >= 0.5 or overlap <= 0.2:
            features.append(proposal_features(gray, box))
            classes.append(int(overlap >= 0.5))

X_train = np.stack(features)
y_train = np.asarray(classes)
if set(y_train) != {0, 1}:
    raise ValueError('Can ca vi du bien so va vi du nhieu trong train.')
print(dict(negatives=int((y_train == 0).sum()), positives=int((y_train == 1).sum()),
           feature_count=X_train.shape[1]))

models = {}
for c in (0.01, 0.1, 1.0):
    model = LinearSVC(C=c, class_weight='balanced', dual=False, max_iter=5000, random_state=SEED)
    models[c] = model.fit(X_train, y_train)

## 3. Chọn cấu hình trên val
Tính precision, recall và F1 trên khung (IoU ≥ 0,5, ghép một-một). Dùng cùng danh sách vùng Sobel cho cả hai phương án. Bộ lọc SVM giữ thứ tự vùng ban đầu và loại các vùng dưới ngưỡng. F1 cao nhất trên val quyết định cấu hình.

In [ ]:
def match_counts(predictions, truth):
    available = set(range(len(truth)))
    matched = 0
    for box, _ in predictions:
        if not available:
            break
        index = max(sorted(available), key=lambda i: iou(box, truth[i]))
        if iou(box, truth[index]) >= IOU_THRESHOLD:
            available.remove(index)
            matched += 1
    return np.array([matched, len(predictions) - matched, len(truth) - matched])

def metrics(counts):
    tp, fp, fn = map(int, counts)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0
    return dict(tp=tp, fp=fp, fn=fn, precision=precision, recall=recall, f1=f1)

def collect_split(paths, split):
    records = []
    feature_rows = []
    for path in paths:
        gray = read_gray(path)
        truth = ground_truth(path, split, gray.shape)
        candidates = proposals(gray)
        start = len(feature_rows)
        feature_rows.extend(proposal_features(gray, box) for box, _ in candidates)
        records.append((truth, candidates, slice(start, len(feature_rows))))
    X = np.stack(feature_rows) if feature_rows else np.empty((0, X_train.shape[1]))
    return records, X

def evaluate(records, scores=None, threshold=None):
    counts = np.zeros(3, dtype=int)
    for truth, candidates, span in records:
        if scores is not None:
            candidates = [box for box, score in zip(candidates, scores[span]) if score >= threshold]
        counts += match_counts(candidates, truth)
    return metrics(counts)

val_records, X_val = collect_split(val_paths, 'val')
val_rows = []
for c, model in models.items():
    scores = model.decision_function(X_val)
    thresholds = sorted(set(np.r_[np.linspace(-1.5, 1.0, 11), np.quantile(scores, [0.02, 0.05, 0.1])]))
    for threshold in thresholds:
        val_rows.append(dict(c=c, threshold=float(threshold),
                             **evaluate(val_records, scores, threshold)))
selected = max(val_rows, key=lambda row: (row['f1'], row['recall'], row['precision']))
print('Baseline val:', evaluate(val_records))
print('SVM val:', selected)

## 4. Đánh giá test và xuất mô hình
Kiểm tra mức giảm vùng khoanh nhầm và mức thay đổi recall. SVM chỉ lọc các vùng đã sinh, nên không thể bù các biển bị bỏ sót ở bước Sobel. Dataset test có ảnh chứa biển. Chưa có nhóm ảnh âm tính riêng để đo báo nhầm theo ảnh tại cổng.

In [ ]:
model = models[selected['c']]
threshold = selected['threshold']
test_records, X_test = collect_split(test_paths, 'test')
test_scores = model.decision_function(X_test)
test_rows = [
    dict(method='sobel', **evaluate(test_records)),
    dict(method='sobel_hog_svm', **evaluate(test_records, test_scores, threshold)),
]
for row in test_rows:
    print(row)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for name, rows in [('validation.csv', val_rows), ('test.csv', test_rows)]:
    with (OUTPUT_DIR / name).open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)

weights_name = 'svm_weights.npz'
np.savez(OUTPUT_DIR / weights_name, weights=model.coef_.ravel(), intercept=model.intercept_.item())
artifact = {
    'proposal': PROPOSAL,
    'max_side': MAX_SIDE,
    'max_boxes': MAX_BOXES,
    'c': selected['c'],
    'threshold': threshold,
    'weights_file': weights_name,
    'eval_iou': IOU_THRESHOLD,
    'train_images': len(train_paths),
    'val_images': len(val_paths),
    'test_images': len(test_paths),
    'smoke_run': SMOKE_RUN,
}
artifact_path = OUTPUT_DIR / 'detector_config.json'
artifact_path.write_text(json.dumps(artifact, indent=2) + '\n', encoding='utf-8')

## 5. Kiểm tra nạp mô hình
Nạp lại file vừa xuất bằng lớp inference của dự án và so với kết quả SVM trên một số ảnh test.

In [ ]:
detector = PlateDetector(artifact_path)
indices = [i for i, (_, candidates, _) in enumerate(test_records) if candidates][:10]
assert indices, 'Khong co ung vien de kiem tra nap mo hinh.'
for index in indices:
    _, candidates, span = test_records[index]
    expected = [box for (box, _), score in zip(candidates, test_scores[span]) if score >= threshold]
    predicted = detector.predict(read_gray(test_paths[index]))
    assert [box for box, _ in predicted] == expected
print(f'Checked saved model on {len(indices)} images.')
print(OUTPUT_DIR.resolve())